# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
#the needed packages 

!pip install duckdb pyarrow huggingface_hub python-dotenv pandas --quiet

In [13]:
# load the token and download the march warehouse files:

import os
import pandas as pd
import duckdb
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download

load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')

from huggingface_hub import list_repo_files

files = list_repo_files(
    "FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)
for f in files:
    print(f)


.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [14]:
# download the fact tables for march (feature month) and april (label month)

fact_mar = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN, repo_type="dataset"
)
fact_apr = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    token=HF_TOKEN, repo_type="dataset"
)

# Download dimension tables (if needed for joins or context)
dim_content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    token=HF_TOKEN, repo_type="dataset"
)

# Load CSV (your local file)
csv = pd.read_csv("../../data/raw/content_refresh_anonymized.csv") 

# Connect DuckDB and register
con = duckdb.connect()
con.execute(f"CREATE VIEW fact_mar AS SELECT * FROM read_parquet('{fact_mar}')")
con.execute(f"CREATE VIEW fact_apr AS SELECT * FROM read_parquet('{fact_apr}')")
con.execute(f"CREATE VIEW dim_content AS SELECT * FROM read_parquet('{dim_content_path}')")
con.register("csv_meta", csv)

# Inspect columns (crucial!)
print("fact_mar columns:", con.sql("DESCRIBE fact_mar").df()['column_name'].tolist())
print("csv_meta columns:", csv.columns.tolist())
print("dim_content columns:", con.sql("DESCRIBE dim_content").df()['column_name'].tolist())

fact_mar columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
csv_meta columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

> One row = one(client_hash_id, content_hash_id) pair for the month of march 2026
>that is the combined search performance of a single content piece, under one client, aggregated over all days in march.
>
>Tables used: fact_content_daily_performance (March & April partitions), dim_content(for static attributes)
>
>Time Window: Observation/feature month = 2026-03. label derived from 2026-04.
>Hold-out month=2026-06(sealed)
>
>Label: growth_state – categorical (growing, declining, stable, recovering) computed from the change in gsc_clicks between March and April.
>
>Excluded: All April fact data except for label construction; fact_content_query_90d and _sample tables; any column whose time window might leak April information.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Bucket 

1. Feature <br>
field1- march_impressions | source- SUM(gsc_impressions) from fact_mar | reason- march data only
field2- march_clicks | source- SUM(gsc_clicks) from fact_mar | reason- march data only
field3- march_avg_position | source- AVG(gsc_avg_position) from fact_mar | reason- march data only
field4- word_count | sorce- dim_content.word_count | reason- static page property
field5- content_age_days | source- DATEDIFF('day', content_created_date, '2026-03-31') from dim_content | reason- Age as of March 31, knowable.
</br>
2. Label <br>
field- growth_state | source-Derived from April vs March gsc_clicks | reason- Strictly future data
</br>
3. context <br>
field- client_hash_id, content_hash_id, month, content_type | source- For joining / filtering / inspection only
</br>
4. Excluded <br>
field- April fact columns, trend_direction/trend_pct from CSV (undefined time window), fact_content_query_90d, any rolling windows that include April | source- Prevent leakage or grain violation
</br>

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

3.1 Grain check - one row per client,content,month

In [15]:
# Aggregate march fact to (client_hash_id, content_hash_id) level
con.execute("""
CREATE VIEW mar_agg AS
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS march_impressions,
    SUM(gsc_clicks) AS march_clicks,
    AVG(gsc_avg_position) AS march_avg_position,
    COUNT(DISTINCT report_date) AS days_present
FROM fact_mar
GROUP BY client_hash_id, content_hash_id
""")

# verify no duplicates for the same (client, content) pair
con.sql("""
SELECT client_hash_id, content_hash_id, COUNT(*) AS cnt
FROM mar_agg
GROUP BY client_hash_id, content_hash_id
HAVING cnt > 1
""").show()


┌────────────────┬─────────────────┬───────┐
│ client_hash_id │ content_hash_id │  cnt  │
│    varchar     │     varchar     │ int64 │
└────────────────┴─────────────────┴───────┘
                   0 rows                 



> here empty results means grain is exactly one ropw per (client, content) for march. as the output is 0 rows

3.2 Row count and date span

In [16]:
con.sql("""
SELECT
    COUNT(*) AS total_client_content_pairs,
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date
FROM fact_mar
""").show()

┌────────────────────────────┬───────────────┬─────────────┐
│ total_client_content_pairs │ earliest_date │ latest_date │
│           int64            │     date      │    date     │
├────────────────────────────┼───────────────┼─────────────┤
│                    9841378 │ 2026-03-01    │ 2026-03-31  │
└────────────────────────────┴───────────────┴─────────────┘



> above observation the row count is 9841378 and the span of the dates

3.3 Availability - filter with IS TRUE

-> We use the boolean column gsc_data_available that exists in the fact table. This tells us whether GSC data was actually present for a given day. We’ll count how many rows survive after filtering.

In [17]:
con.sql("""
SELECT COUNT(*) AS rows_with_gsc_data
FROM fact_mar
WHERE gsc_data_available IS TRUE
""").show()

┌────────────────────┐
│ rows_with_gsc_data │
│       int64        │
├────────────────────┤
│            3611061 │
└────────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

-> We built a feature frame by joining mar_agg and dim_content to get the static attributes.<br>
all feature are knowable on march 31,2026.</br>

In [18]:
# Build the honest feature frame
features_df = con.sql("""
SELECT
    m.march_impressions,
    m.march_clicks,
    m.march_avg_position,
    d.word_count,
    DATEDIFF('day', d.content_created_date, DATE '2026-03-31') AS content_age_days
FROM mar_agg m
JOIN dim_content d ON m.content_hash_id = d.content_hash_id
                  AND m.client_hash_id = d.client_hash_id
""").df()

features_df.head()

,march_impressions,march_clicks,march_avg_position,word_count,content_age_days
0,5586.0,36.0,2.518175,2893,195
1,692.0,0.0,2.457830,<NA>,195
2,9437.0,20.0,6.977657,2385,195
3,250.0,2.0,9.535076,2235,195
4,90.0,0.0,10.645679,<NA>,195


the above feature frame details
1. march_impressions - Sum of daily gsc_impressions for March 1‑31 — all data exists by March 31.
2. march_clicks - Sum of daily gsc_clicks for March — same period.
3. march_avg_position - Average of gsc_avg_position across March days.
4. word_count - Static attribute stored in dim_content, never changes.
5. content_age_days - Computed as days from content_created_date to March 31 — fixed value at the decision moment.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.